# Notebook 02 · Carbon storage change, 1990–2018, and how far to trust it

**Exercise:** Carbon storage change in mainland Portugal, 1990–2018
**Companion to:** Pedro Cabral, *Monitoring Dynamics in Ecosystem Service Indicators with Remote Sensing* (Elsevier)

This notebook runs the InVEST Carbon Storage and Sequestration model on the land-cover maps prepared in Notebook 01, then goes beyond the Barcelona tutorial deliverables in three ways: it traces the change back to the land-cover transitions that caused it, expresses its value as a range rather than a single number, and maps where the result is robust to uncertainty in the carbon table.

| Step | What you do | Book section |
|---|---|---|
| 1 | Inspect the carbon table and its assumptions | 4.6.3 |
| 2 | Run InVEST for 1990 and 2018 | 4.6.2 |
| 3 | Summarize storage and change (tutorial deliverables) | 4.6.4 |
| 4 | Attribute change to transitions and carbon pools | 4.2.3, 4.7.1, 5.7 |
| 5 | Aggregate by municipality | 3.5 |
| 6 | Express value as a range | 6.4 |
| 7 | Ensemble uncertainty and confidence bands | 8.3, 8.6 (Table 8.1) |

Run Notebook 01 first.

In [ ]:
from importlib.metadata import version
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from rasterio.features import rasterize

import es_utils as es

DATA, OUT = es.DATA_DIR, es.OUT_DIR
OUT.mkdir(exist_ok=True)
CLC90, CLC18 = DATA / "clc1990_portugal.tif", DATA / "clc2018_portugal.tif"
TABLE = DATA / "carbon_pools.csv"

lc90, prof = es.read_clc(CLC90)
lc18, _ = es.read_clc(CLC18)
AREA_HA = es.pixel_area_ha(prof)
valid = (lc90 >= 0) & (lc18 >= 0)

## Step 1 · The carbon table and what it assumes

> **Book link · Section 4.6.3.** The model's result depends on the carbon table more than on anything else. Two properties matter here:
> - **One table for both years.** Because 1990 and 2018 use identical values, every change in carbon comes from a change in land-cover class. That is what makes the comparison interpretable, and also what limits it.
> - **One value per class.** All pixels of a class store the same carbon, whatever their age, condition or management.

The values are carbon densities in t C ha⁻¹ for four pools: aboveground biomass, belowground biomass, soil organic carbon and dead organic matter.

In [ ]:
table = pd.read_csv(TABLE)
table["c_total"] = table[es.POOLS].sum(axis=1)
present = set(np.unique(lc90[lc90 >= 0])) | set(np.unique(lc18[lc18 >= 0]))
shown = table[table["lucode"].isin(present)].sort_values("c_total", ascending=False)

fig, ax = plt.subplots(figsize=(8, 0.28 * len(shown) + 1))
left = np.zeros(len(shown))
for pool, colour in zip(es.POOLS, ["#1b7837", "#7fbf7b", "#8c510a", "#dfc27d"]):
    ax.barh(shown["lulc_name"], shown[pool], left=left, color=colour, label=pool)
    left += shown[pool].to_numpy()
ax.invert_yaxis()
ax.set_xlabel("Carbon density (t C ha⁻¹)")
ax.set_title("Carbon pools by CLC class present in the study area")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()

**Look at *Burnt areas* (code 33).** Its soil carbon is 1.52 t C ha⁻¹, against 73–120 for forests and about 110 for transitional woodland-shrub. Any forest or shrubland mapped as burnt in 2018 will therefore lose roughly 70–120 t C ha⁻¹ in this model, mostly *soil* carbon. A fire removes much of the vegetation and litter, but most soil organic carbon remains. Keep this in mind in Step 4; Notebook 03 checks what the satellite record says about the same pixels.

> **Book link · Section 4.7.1.** This is a concrete case of treating a land-cover label as if it measured the service: the label changes overnight, the stock does not.

## Step 2 · Run the InVEST carbon model

> **Book link · Section 4.6.2 (workflow).** Inputs: a baseline land-cover map (1990), an alternative map (2018) and the carbon table. Outputs: carbon storage for each year and the change between them, all in t C ha⁻¹ per pixel.

This uses InVEST's Python API (`natcap.invest` 3.15 or later). The same run can be done in the InVEST Workbench with identical inputs.

In [ ]:
from natcap.invest import carbon

major, minor = (int(x) for x in version("natcap.invest").split(".")[:2])
assert (major, minor) >= (3, 15), "This notebook needs natcap.invest 3.15 or later."

WORKSPACE = OUT / "invest_carbon"
args = {
    "workspace_dir": str(WORKSPACE),
    "results_suffix": "",
    "n_workers": -1,
    "lulc_bas_path": str(CLC90),
    "calc_sequestration": True,
    "lulc_alt_path": str(CLC18),
    "carbon_pools_path": str(TABLE),
    "do_valuation": False,
}
carbon.execute(args)
print(sorted(p.name for p in WORKSPACE.glob("*.tif")))

## Step 3 · Carbon storage and change: the tutorial deliverables

Because each pixel is exactly 1 ha, the sum of pixel values in t C ha⁻¹ is also the total stock in tonnes.

In [ ]:
def read_output(name):
    with rasterio.open(WORKSPACE / name) as src:
        return src.read(1, masked=True).filled(np.nan).astype(np.float32)

c90 = read_output("c_storage_bas.tif")
c18 = read_output("c_storage_alt.tif")
dc = read_output("c_change_bas_alt.tif")
mask = valid & ~np.isnan(c90) & ~np.isnan(c18)

summary = pd.Series({
    "Mean carbon storage 1990 (t C/ha)": np.mean(c90[mask]),
    "Mean carbon storage 2018 (t C/ha)": np.mean(c18[mask]),
    "Absolute change (t C/ha)": np.mean(c18[mask]) - np.mean(c90[mask]),
    "Relative change (%)": 100 * (np.mean(c18[mask]) - np.mean(c90[mask])) / np.mean(c90[mask]),
    "Total stock 1990 (Tg C)": np.sum(c90[mask]) * AREA_HA / 1e6,
    "Total stock 2018 (Tg C)": np.sum(c18[mask]) * AREA_HA / 1e6,
    "Net change (Tg C)": np.sum(dc[mask]) * AREA_HA / 1e6,
    "Area with any change (kha)": np.sum(mask & (dc != 0)) * AREA_HA / 1e3,
})
summary.to_csv(OUT / "02_summary.csv", header=["value"])
display(summary.round(2).to_frame("value"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 7))
vmax = np.nanpercentile(c90[mask], 99)
es.plot_continuous(axes[0], np.where(mask, c90, np.nan), prof, "Carbon storage, 1990", "t C ha⁻¹", vmin=0, vmax=vmax)
es.plot_continuous(axes[1], np.where(mask, c18, np.nan), prof, "Carbon storage, 2018", "t C ha⁻¹", vmin=0, vmax=vmax)
change_map = np.where(mask & (dc != 0), dc, np.nan)
es.plot_continuous(axes[2], change_map, prof, "Change 1990–2018 (changed pixels only)", "t C ha⁻¹",
                   cmap="RdBu", diverging=True, vmax=150)
plt.tight_layout()
fig.savefig(OUT / "02_carbon_maps.png", dpi=150)

## Step 4 · Which transitions and which pools produced the change?

> **Book link · Section 4.2.3 (the indicator–driver–benefit chain) and Section 5.7 (attribution).** A net figure hides gains and losses that partly cancel. Decomposing it by transition shows what the indicator is actually responding to. The decomposition here is exact, because in this model every pixel's change is fully determined by its two class labels. That does not make it causal: it tells you which *label changes* drove the number, not which *processes* did.

In [ ]:
g90, g18 = es.to_groups(lc90), es.to_groups(lc18)
n = len(es.GROUP_ORDER)
chg = mask & (lc90 != lc18)
pair = g90[chg].astype(np.int64) * n + g18[chg]
area_k = np.bincount(pair, minlength=n * n) * AREA_HA / 1e3
dc_tg = np.bincount(pair, weights=dc[chg].astype(np.float64), minlength=n * n) * AREA_HA / 1e6

trans = pd.DataFrame({
    "from (1990)": np.repeat(es.GROUP_ORDER, n),
    "to (2018)": np.tile(es.GROUP_ORDER, n),
    "area (kha)": area_k,
    "carbon change (Tg C)": dc_tg,
})
trans = trans[trans["area (kha)"] > 0]
trans["share of gross change (%)"] = 100 * trans["carbon change (Tg C)"].abs() / trans["carbon change (Tg C)"].abs().sum()
trans = trans.sort_values("carbon change (Tg C)")
trans.to_csv(OUT / "02_change_by_transition.csv", index=False)

print("Largest losses:")
display(trans[trans["carbon change (Tg C)"] < 0].head(6).round(2))
print("Largest gains:")
display(trans[trans["carbon change (Tg C)"] > 0].iloc[::-1].head(6).round(2))

In [ ]:
pool_change = {}
for pool in es.POOLS:
    p90, p18 = es.lookup(lc90, table, pool), es.lookup(lc18, table, pool)
    pool_change[pool] = np.nansum((p18 - p90)[mask]) * AREA_HA / 1e6
pool_change = pd.Series(pool_change, name="net change (Tg C)")
display(pool_change.round(2).to_frame())
share_soil = 100 * abs(pool_change["c_soil"]) / pool_change.abs().sum()
print(f"Soil organic carbon accounts for {share_soil:.0f}% of the absolute change summed over pools.")

**Interpretation boundaries** (Section 4.6.4)
- **Report patterns, not stocks.** The spatial pattern of gains and losses is more robust than the absolute tonnes, which inherit every bias in the carbon table.
- **The model has no time.** A class change in the map becomes a full stock change at once. Real soil carbon responds over decades, so the soil share above is the part of the result most likely to be overstated, in both directions.
- **No within-class change.** Forest that thickened or degraded without changing class contributes nothing here. Notebook 03 looks for exactly these pixels.

## Step 5 · Aggregate by municipality

> **Book link · Section 3.5 (spatial aggregation).** Decisions are made for administrative units, not pixels. Aggregation is also where the modifiable areal unit problem enters: the same pixels summarized over different units can tell different stories.

Here each municipality polygon is rasterized onto the CLC grid, so every 1 ha pixel belongs to exactly one municipality and no area-weighting of partial pixels is needed. The municipality layer is produced by `scripts/clip_clc.py --caop`.

In [ ]:
MUNI = DATA / "municipalities.gpkg"
if not MUNI.exists():
    print("municipalities.gpkg not found: run scripts/clip_clc.py with --caop to create it. Skipping this step.")
else:
    muni = gpd.read_file(MUNI).to_crs(prof["crs"]).reset_index(drop=True)
    zone = rasterize(((geom, i + 1) for i, geom in enumerate(muni.geometry)),
                     out_shape=lc90.shape, transform=prof["transform"], fill=0, dtype="int32")
    z = zone[mask]
    nz = len(muni) + 1
    pixels = np.bincount(z, minlength=nz)
    muni["area_kha"] = pixels[1:] * AREA_HA / 1e3
    muni["net_change_kt_C"] = np.bincount(z, weights=dc[mask].astype(np.float64), minlength=nz)[1:] * AREA_HA / 1e3
    muni["mean_change_t_ha"] = muni["net_change_kt_C"] * 1e3 / (muni["area_kha"] * 1e3).replace(0, np.nan)
    muni.drop(columns="geometry").to_csv(OUT / "02_change_by_municipality.csv", index=False)

    fig, ax = plt.subplots(figsize=(6, 8))
    lim = np.nanpercentile(np.abs(muni["mean_change_t_ha"]), 98)
    muni.plot(column="mean_change_t_ha", cmap="RdBu", vmin=-lim, vmax=lim, ax=ax, legend=True,
              legend_kwds={"label": "Mean carbon change (t C ha⁻¹)", "shrink": 0.6},
              edgecolor="grey", linewidth=0.2)
    ax.set_title("Mean carbon storage change by municipality, 1990–2018")
    ax.set_axis_off()
    display(muni.nsmallest(10, "net_change_kt_C")[["name", "area_kha", "net_change_kt_C", "mean_change_t_ha"]].round(1))

## Step 6 · Express the value as a range

> **Book link · Section 6.4 (valuation).** A monetary figure depends on a price that is itself a choice. Reporting the result under several prices makes that choice visible instead of hiding it in one number.

The prices below are illustrative and should be replaced with those relevant to your question: a market-based price of the kind seen in the EU Emissions Trading System, and higher values of the kind used in social-cost-of-carbon estimates. Carbon is converted to CO₂ with the molar-mass ratio 44/12. No discounting is applied, since this is a difference between two stocks rather than a flow over time.

In [ ]:
CO2_PER_C = 44 / 12
prices_eur_per_t_co2 = {"low (25 €/t CO₂)": 25, "market-like (80 €/t CO₂)": 80, "high (200 €/t CO₂)": 200}

net_tg_c = summary["Net change (Tg C)"]
gross_loss_tg_c = trans.loc[trans["carbon change (Tg C)"] < 0, "carbon change (Tg C)"].sum()
value = pd.DataFrame({
    label: {"net change (M€)": net_tg_c * CO2_PER_C * p, "gross losses only (M€)": gross_loss_tg_c * CO2_PER_C * p}
    for label, p in prices_eur_per_t_co2.items()
})
display(value.round(0))
print("Tg C × 44/12 = Tg CO₂; Tg CO₂ × €/t = M€.")

## Step 7 · How confident can we be in the change map?

> **Book link · Section 8.3 (sensitivity analysis), Section 8.6 (worked example) and Table 8.1.** The carbon table is uncertain. Rather than report one map, run the model many times with plausible variations of the table and see where the answer holds.

**A shortcut that is exact, not approximate.** The InVEST carbon model is a lookup: each pixel's change depends only on its 1990 class, its 2018 class and the table. So instead of rerunning InVEST hundreds of times, we can compute the change for each *pair of classes* in each realization and map it back. The first cell checks that the lookup reproduces InVEST's output exactly.

In [ ]:
dc_lookup = es.lookup(lc18, table, "c_total") - es.lookup(lc90, table, "c_total")
max_diff = np.nanmax(np.abs(dc_lookup[mask] - dc[mask]))
print(f"Largest difference between lookup and InVEST change: {max_diff:.2e} t C/ha")
assert max_diff < 1e-3

**Uncertainty assumption.** Each pool value of each class is multiplied by an independent random factor drawn uniformly between 0.75 and 1.25 (±25%). This is illustrative: replace it with class-specific ranges from your sources where they exist. The confidence bands then follow the coefficient-of-variation (CV) thresholds of the book's worked example (Table 8.1): **high** confidence below 0.15, **moderate** from 0.15 to 0.35, **low** above 0.35.

The notebook also records **sign agreement**: the share of realizations in which a pixel changes in the same direction as the ensemble mean. CV and sign agreement answer different questions (Section 8.2): how precisely we know the magnitude, and how robust the direction is.

In [ ]:
N_REAL, SPREAD, SEED = 300, 0.25, 2026
rng = np.random.default_rng(SEED)

codes_all = table["lucode"].to_numpy()
pos = {c: i for i, c in enumerate(codes_all)}
base = table[es.POOLS].to_numpy(np.float64)                         # (classes, pools)
factors = rng.uniform(1 - SPREAD, 1 + SPREAD, (N_REAL,) + base.shape)
totals = (base[None] * factors).sum(axis=2)                          # (realizations, classes)

pairs, inverse = np.unique(lc90[chg].astype(np.int32) * 256 + lc18[chg], return_inverse=True)
fi = np.array([pos[p // 256] for p in pairs])
ti = np.array([pos[p % 256] for p in pairs])
ens = totals[:, ti] - totals[:, fi]                                  # (realizations, pairs)

mean_p = ens.mean(axis=0)
cv_p = ens.std(axis=0) / np.abs(mean_p)
agree_p = (np.sign(ens) == np.sign(mean_p)).mean(axis=0)
print(f"{len(pairs)} distinct class transitions, {N_REAL} realizations each.")

In [ ]:
BANDS = ["No change", "High confidence", "Moderate confidence", "Low confidence"]
band_p = np.where(cv_p < 0.15, 1, np.where(cv_p <= 0.35, 2, 3))

band = np.full(lc90.shape, -1, np.int8)
band[mask] = 0
band[chg] = band_p[inverse]
ens_mean = np.where(mask, 0.0, np.nan).astype(np.float32)
ens_mean[chg] = mean_p[inverse]
agree = np.full(lc90.shape, np.nan, np.float32)
agree[chg] = agree_p[inverse]

direction = np.where(ens_mean > 0, "gain", np.where(ens_mean < 0, "loss", "none"))
tab = pd.crosstab(pd.Series(np.array(BANDS)[band[mask]], name="confidence band"),
                  pd.Series(direction[mask], name="direction")) * AREA_HA / 1e3
display(tab.reindex(BANDS).fillna(0).round(1).rename(columns=lambda c: f"{c} (kha)"))
print(f"Changed area where fewer than 90% of realizations agree on the direction: "
      f"{np.sum(chg & (agree < 0.9)) * AREA_HA / 1e3:,.1f} kha")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))
es.plot_continuous(axes[0], np.where(chg, ens_mean, np.nan), prof,
                   "Ensemble mean change (changed pixels)", "t C ha⁻¹", cmap="RdBu", diverging=True, vmax=150)
band_cmap = ListedColormap(["#f0f0f0", "#1a9850", "#fee08b", "#d73027"])
band_cmap.set_bad("white")
axes[1].imshow(np.ma.masked_less(band, 0), cmap=band_cmap, vmin=-0.5, vmax=3.5,
               extent=es.extent(prof), interpolation="nearest")
axes[1].legend(handles=[Patch(color=c, label=b) for c, b in zip(band_cmap.colors, BANDS)],
               loc="lower left", fontsize=8)
axes[1].set_title("Confidence band (Table 8.1 thresholds)")
axes[1].set_xticks([]); axes[1].set_yticks([])
plt.tight_layout()
fig.savefig(OUT / "02_confidence_map.png", dpi=150)

trans_conf = pd.DataFrame({"from": [table.loc[pos[p // 256], "lulc_name"] for p in pairs],
                           "to": [table.loc[pos[p % 256], "lulc_name"] for p in pairs],
                           "area_ha": np.bincount(inverse) * AREA_HA,
                           "mean_change_t_ha": mean_p, "cv": cv_p, "sign_agreement": agree_p})
trans_conf["band"] = np.array(BANDS)[band_p]
trans_conf.sort_values("area_ha", ascending=False).to_csv(OUT / "02_transition_confidence.csv", index=False)
display(trans_conf.sort_values("area_ha", ascending=False).head(12).round(2))

**Reading the confidence map**
- Confidence here is a property of the *transition*, because the model is a lookup. Forest becoming urban is a large, unambiguous loss in every realization. A change between two classes with similar carbon totals can flip sign from one realization to the next.
- High confidence means the result is stable *under the assumed table uncertainty*. It does not cover errors in the land-cover maps themselves, the missing time dimension, or within-class change (Section 8.6's closing caveat applies directly).
- A natural extension, as in Chapter 8, is to also perturb the land-cover maps, for example by reassigning a share of pixels according to CLC's published thematic accuracy.

## Barcelona deliverables checklist

- [ ] Maps of carbon storage 1990, 2018 and change, with legend, units, scale and title (Step 3; `outputs/02_carbon_maps.png`)
- [ ] Mean storage 1990 and 2018, absolute and relative change (Step 3; `outputs/02_summary.csv`)
- [ ] Interpretation (≤200 words): direction of change, where, and which land-cover changes explain it (Step 4)
- [ ] Methodological note (≤100 words): operations, assumptions, problems (Notebook 01 and Steps 1–2)

## Discussion questions

1. More than half of the absolute change may come from the soil pool. Rewrite the summary in Step 3 so that it is honest about this without discarding the result. *(Sections 4.6.4, 4.7.1)*
2. Pick the largest transition in the loss table. Name one process that could cause it and one mapping artefact that could produce the same label change. How would you tell them apart? *(Sections 3.2, 5.7)*
3. Which municipalities would a planner single out from the municipal map, and which would they miss if they looked only at net change rather than gross losses? *(Sections 3.5, 9.3)*
4. A colleague proposes reporting the high-price valuation in a policy brief because "it makes the point more strongly". What would you advise, and why? *(Section 6.4)*
5. Replace the ±25% assumption with ±50% for soil carbon only. Which transitions move to a lower confidence band, and what does that tell you about where better field data would help most? *(Sections 8.3, 8.6)*

**Next:** Notebook 03 uses a MODIS vegetation time series to look inside the land-cover classes that this model treats as uniform.

---
*Data: CORINE Land Cover, © European Union, Copernicus Land Monitoring Service, European Environment Agency (EEA). Model: InVEST Carbon Storage and Sequestration, Natural Capital Project, Stanford University.*